# RAG-QPP: Query Performance Prediction — Notebook 2
**RQ2**: How well do Random Forest, XGBoost, and LightGBM predict query-level MRR@10?

Reproduces Table 4 of the paper.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr, spearmanr, kendalltau

from src.features  import extract_features_batch, normalize_features, FEATURE_NAMES
from src.models    import train_all_models, predict, get_feature_importance
from src.evaluate  import compute_correlations

np.random.seed(42)
print('Imports OK')

## 1. Load pre-computed features (or generate from retriever output)

In [ ]:
# ── Option A: load saved numpy arrays ────────────────────────────────────────
# X = np.load('outputs/predictions/X_msmarco_passage_dense.npy')
# y = np.load('outputs/predictions/y_mrr10_msmarco_passage.npy')

# ── Option B: synthetic demo data (replace with real features) ───────────────
N = 500
X = np.random.randn(N, 12).astype(np.float32)
y = np.clip(np.random.beta(2, 5, N).astype(np.float32), 0, 1)

# Train / test split
split = int(0.8 * N)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

X_train_n, X_test_n, scaler = normalize_features(X_train, X_test)
print(f'Train: {X_train.shape}  Test: {X_test.shape}')

## 2. Train all three QPP models

In [ ]:
models = train_all_models(X_train_n, y_train, cv=True)
print('Training complete.')

## 3. Evaluate: Pearson r / Spearman ρ / Kendall τ  (Table 4)

In [ ]:
rows = []
for name, model in models.items():
    y_pred = predict(model, X_test_n)
    corr   = compute_correlations(y_pred, y_test)
    rows.append({
        'Model':       name,
        'Pearson r':   round(corr['pearson_r'],    4),
        'Spearman ρ':  round(corr['spearman_rho'], 4),
        "Kendall τ":   round(corr['kendall_tau'],  4),
        'p (Pearson)': round(corr['pearson_p'],    4),
    })

df_corr = pd.DataFrame(rows)
display(df_corr)

## 4. Scatter plot: predicted QPP vs true MRR@10  (Figure 2)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
colors = ['#2196F3', '#4CAF50', '#FF9800']

for ax, (name, model), color in zip(axes, models.items(), colors):
    y_pred = predict(model, X_test_n)
    r, _   = pearsonr(y_pred, y_test)
    ax.scatter(y_pred, y_test, alpha=0.4, s=15, color=color)
    m, b = np.polyfit(y_pred, y_test, 1)
    xs = np.linspace(y_pred.min(), y_pred.max(), 100)
    ax.plot(xs, m*xs + b, 'r-', lw=2)
    ax.set_title(f'{name}\nPearson r = {r:.4f}', fontsize=11)
    ax.set_xlabel('Predicted QPP Score')
    ax.set_ylabel('True MRR@10')
    ax.grid(alpha=0.3)

plt.suptitle('Predicted QPP Scores vs True MRR@10', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('outputs/figures/scatter_qpp_vs_mrr.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Feature Importance  (Figure 5)

In [ ]:
importance = get_feature_importance(models['random_forest'])
imp_df = pd.DataFrame(
    sorted(importance.items(), key=lambda x: x[1]),
    columns=['Feature', 'Importance']
)

plt.figure(figsize=(8, 5))
colors_imp = ['#FF7043' if 'sim' in f or 'emb' in f else '#78909C' for f in imp_df['Feature']]
plt.barh(imp_df['Feature'], imp_df['Importance'], color=colors_imp)
plt.xlabel('Relative Feature Importance')
plt.title('Per-Feature Contribution to QPP Prediction (MS MARCO Passage)')
plt.tight_layout()
plt.savefig('outputs/figures/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nTop features:')
print(imp_df.tail(5).to_string(index=False))